# ChemBreak V9 Cloud
## Google Colab + Vertex AI

This notebook clones your GitHub repository and uses only the `ChemBreak_V9_Cloud` folder.

V9 is independent of every earlier ChemBreak code folder. Removing an older version later will not affect this folder.


## 1. Python setup


In [ ]:
from pathlib import Path
import subprocess
import sys
import json
import os
import shutil

PROJECT_ID = "rs-foundsecft-mghasemi"

print("Python setup: OK")
print("Project:", PROJECT_ID)


## 2. Authenticate standard Colab to Google Cloud

Open the displayed Google URL, sign in with the account that has access to the project, then paste the verification code back into Colab.


In [ ]:
!gcloud auth application-default login --no-launch-browser


## 3. Attach the Google Cloud project to ADC


In [ ]:
!gcloud auth application-default set-quota-project rs-foundsecft-mghasemi
!gcloud config set project rs-foundsecft-mghasemi

import google.auth

credentials, detected_project = google.auth.default(
    scopes=["https://www.googleapis.com/auth/cloud-platform"]
)

print("Authentication: OK")
print("Detected project:", detected_project)
print("Quota project:", credentials.quota_project_id)
print("Credential type:", type(credentials).__name__)


## 4. Mount Google Drive for persistent checkpoints


In [ ]:
USE_GOOGLE_DRIVE = True

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    STORAGE_ROOT = Path("/content/drive/MyDrive/ChemBreak_V9")
else:
    STORAGE_ROOT = Path("/content/ChemBreak_V9")

STORAGE_ROOT.mkdir(parents=True, exist_ok=True)
print("Persistent storage root:", STORAGE_ROOT)


## 5. Clone or refresh the GitHub repository

This notebook uses only `ChemBreak_V9_Cloud`.


In [ ]:
REPO_URL = "https://github.com/Jollychuks/ChemBreak.git"
REPO_ROOT = Path("/content/ChemBreak_repo")
PROJECT_SUBDIR = "ChemBreak_V9_Cloud"

if (REPO_ROOT / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"], check=True)
elif REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)

PROJECT_DIR = REPO_ROOT / PROJECT_SUBDIR
if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(
        f"{PROJECT_SUBDIR} was not found in GitHub. "
        "Upload the complete V9 folder to your repository first."
    )

PIPELINE = PROJECT_DIR / "scripts" / "chembreak_v9_cloud.py"
CONFIG_SOURCE = PROJECT_DIR / "config" / "run_config.json"

print("V9 folder:", PROJECT_DIR)
print("Pipeline:", PIPELINE)


## 6. Install V9 requirements


In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_DIR / "requirements.txt")],
    check=True
)
print("V9 requirements installed.")


## 7. Choose the run

Start with `test`.

- test: 9 final tasks
- pilot: 100 final tasks plus reserve assignments
- production: 500 final tasks plus reserve assignments


In [ ]:
RUN_TYPE = "test"
GCS_OUTPUT_URI = ""

RUNTIME_DIR = Path("/content/ChemBreak_V9_runtime")
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
RUNTIME_CONFIG = RUNTIME_DIR / f"run_config_{RUN_TYPE}.json"

cfg = json.loads(CONFIG_SOURCE.read_text(encoding="utf-8"))
cfg["run_type"] = RUN_TYPE
cfg["project_id"] = PROJECT_ID
cfg["gcs_output_uri"] = GCS_OUTPUT_URI
RUNTIME_CONFIG.write_text(json.dumps(cfg, indent=2), encoding="utf-8")

OUTPUT_DIR = STORAGE_ROOT / "outputs" / RUN_TYPE
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def run_stage(stage):
    command = [
        sys.executable, "-u", str(PIPELINE),
        "--stage", stage,
        "--project-dir", str(PROJECT_DIR),
        "--config", str(RUNTIME_CONFIG),
        "--output-dir", str(OUTPUT_DIR),
    ]
    print(f"\n===== {stage.upper()} =====\n", flush=True)
    subprocess.run(command, check=True)
    print(f"\nCompleted: {stage}", flush=True)

print("Run type:", RUN_TYPE)
print("Output directory:", OUTPUT_DIR)


## 8. Preflight Vertex AI model access


In [ ]:
run_stage("preflight")

import pandas as pd
from IPython.display import display

display(pd.read_csv(OUTPUT_DIR / "preflight_models.csv"))


## 9. Bootstrap fresh V9 source provenance


In [ ]:
run_stage("bootstrap")


## 10. Build the fresh V9 assignment plan


In [ ]:
run_stage("plan")


## 11. Inspect V9 coverage before generation


In [ ]:
plan = pd.read_csv(OUTPUT_DIR / "assignments_v9.csv")

display(plan.head(20))
print("Assignments:", len(plan))

print("\nHC coverage")
display(plan["hc_id"].value_counts().sort_index())

print("\nHD coverage")
display(plan["hd_id"].value_counts().sort_index())

print("\nOT coverage")
display(plan["ot_id"].value_counts().sort_index())


## 12. Generate the candidate pool


In [ ]:
run_stage("generate")


## 13. Deterministic validation


In [ ]:
run_stage("validate")


## 14. Repair invalid candidates


In [ ]:
run_stage("repair")


## 15. Blind judging


In [ ]:
run_stage("judge")


## 16. Refill unresolved assignments and judge again


In [ ]:
run_stage("refill")
run_stage("judge")


## 17. Finalize and inspect the V9 task bank


In [ ]:
run_stage("finalize")
run_stage("status")

for name in [
    "run_summary.json",
    "coverage_report.csv",
    "diversity_report.csv",
    "final_task_bank.csv",
]:
    path = OUTPUT_DIR / name
    print("\n", name)
    if path.suffix == ".json" and path.exists():
        print(path.read_text(encoding="utf-8"))
    elif path.exists():
        display(pd.read_csv(path).head(25))
    else:
        print("not written")


## 18. Create a persistent checkpoint ZIP


In [ ]:
summary = json.loads((OUTPUT_DIR / "run_summary.json").read_text(encoding="utf-8"))
label = summary["completion_label"]

archive = shutil.make_archive(
    str(STORAGE_ROOT / f"ChemBreak_V9_{RUN_TYPE}_{label}"),
    "zip",
    root_dir=str(OUTPUT_DIR)
)

print("Checkpoint ZIP:", archive)
